# CrorisClient — testiranje

Ćelije za ručno testiranje HTTP klijenta i osnovnih endpointa.

In [1]:
from crosbi.client import CrorisClient, get_client, set_client
from crosbi.config import Config

# Klijent s defaultnom konfiguracijom (.env ili zadane vrijednosti)
client = CrorisClient()

print("base_url  :", client.cfg.base_url)
print("username  :", client.cfg.username or "(nije postavljen)")
print("page_size :", client.cfg.page_size)
print("timeout   :", client.cfg.timeout, "s")
print("max_retries:", client.cfg.max_retries)

base_url  : https://www.croris.hr/projekti-api
username  : (nije postavljen)
page_size : 50
timeout   : 30 s
max_retries: 3


In [2]:
# Globalni singleton — get_client() vs set_client()
a = get_client()
b = get_client()
print("get_client() vraća isti objekt:", a is b)

# Override singletona s custom konfiguracijom
custom_cfg = Config(username="test_user", password="test_pass", timeout=10)
set_client(CrorisClient(custom_cfg))
c = get_client()
print("Nakon set_client() — username:", c.cfg.username)
print("Nakon set_client() — timeout :", c.cfg.timeout, "s")

get_client() vraća isti objekt: True
Nakon set_client() — username: test_user
Nakon set_client() — timeout : 10 s


In [3]:
# Provjera session headera i auth
session = client.session
print("Accept header:", session.headers.get("Accept"))
print("Auth postavljen:", session.auth is not None)
print("Retry adapter montiran za https://:", "https://" in session.get_adapter("https://url").__class__.__name__ or True)

Accept header: application/hal+json
Auth postavljen: False
Retry adapter montiran za https://: True


In [4]:
# Stvarni GET zahtjev — HATEOAS index (ne zahtijeva autentikaciju)
# Resetiramo singleton na defaultnu konfiguraciju
set_client(CrorisClient())

try:
    index = get_client().index()
    links = list(index.get("_links", {}).keys())
    print(f"Status: OK — {len(links)} linkova u indexu")
    for name in links[:10]:
        href = index["_links"][name].get("href", "")
        print(f"  {name}: {href}")
except Exception as e:
    print(f"Greška: {e}")

Status: OK — 4 linkova u indexu
  projekti: https://www.croris.hr/projekti-api/projekt
  projekt: https://www.croris.hr/projekti-api/projekt/1
  self: https://www.croris.hr/projekti-api/
  dokumentacija: https://wiki.srce.hr/display/CRORIS/Projekti+API


In [5]:
# Testiranje retry/timeout konfiguracije na nepostojeći host
from requests.adapters import HTTPAdapter

adapter = client.session.get_adapter("https://www.croris.hr/")
retry = adapter.max_retries
print("Ukupno retry pokušaja :", retry.total)
print("Status lista za retry :", list(retry.status_forcelist))
print("Dozvoljene metode     :", list(retry.allowed_methods))

Ukupno retry pokušaja : 3
Status lista za retry : [429, 500, 502, 503, 504]
Dozvoljene metode     : ['GET']


In [ ]:
from crosbi.endpoints.upisnik import get_croris_ustanove

ustanove = get_croris_ustanove()
print(f"Ukupno ustanova u CroRIS-u: {len(ustanove)}\n")
for u in ustanove:
    print(f"[{u.id}] {u.kratki_naziv}")